In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
data = pd.read_csv("Churn_Modelling.csv")

print("First five rows:")
print(data.head())

print("\nDataset shape:", data.shape)

First five rows:
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       

In [4]:
# Step 3: Separate features and target

# Remove unnecessary columns
X = data.drop(['RowNumber', 'CustomerId', 'Surname', 'Exited'], axis=1)

# Target variable
y = data['Exited']

print("\nFeatures:")
print(X.head())

print("\nTarget:")
print(y.head())



Features:
   CreditScore Geography  Gender  Age  Tenure    Balance  NumOfProducts  \
0          619    France  Female   42       2       0.00              1   
1          608     Spain  Female   41       1   83807.86              1   
2          502    France  Female   42       8  159660.80              3   
3          699    France  Female   39       1       0.00              2   
4          850     Spain  Female   43       2  125510.82              1   

   HasCrCard  IsActiveMember  EstimatedSalary  
0          1               1        101348.88  
1          0               1        112542.58  
2          1               0        113931.57  
3          0               0         93826.63  
4          1               1         79084.10  

Target:
0    1
1    0
2    1
3    0
4    0
Name: Exited, dtype: int64


In [5]:
# Step 4: Encode categorical variables

# Geography and Gender are categorical features

preprocessor = ColumnTransformer(
    transformers=[
        ('geo', OneHotEncoder(handle_unknown='ignore',
                              drop='first'), ['Geography']),
        ('gender', OneHotEncoder(handle_unknown='ignore',
                                 drop='first'), ['Gender']),
        ('num', StandardScaler(),
         ['CreditScore', 'Age', 'Tenure', 'Balance',
          'NumOfProducts', 'HasCrCard',
          'IsActiveMember', 'EstimatedSalary'])
    ]
)


In [6]:
# Step 5: Divide data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
# Step 6: Normalize and encode data

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

print("\nTraining data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)


Training data shape: (8000, 11)
Testing data shape: (2000, 11)


In [8]:
# Step 7: Initialize and build the ANN model

model = Sequential()

# Input layer and first hidden layer
model.add(Dense(64, activation='relu',
                input_shape=(X_train.shape[1],)))

# Second hidden layer
model.add(Dense(32, activation='relu'))

# Dropout to reduce overfitting
model.add(Dropout(0.3))

# Output layer
model.add(Dense(1, activation='sigmoid'))

C:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
# Step 8: Compile the model

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()



Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │             768 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Step 9: Train the model

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7207 - loss: 0.5475 - val_accuracy: 0.8256 - val_loss: 0.4147
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8221 - loss: 0.4125 - val_accuracy: 0.8475 - val_loss: 0.3860
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8353 - loss: 0.3984 - val_accuracy: 0.8600 - val_loss: 0.3588
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8443 - loss: 0.3815 - val_accuracy: 0.8694 - val_loss: 0.3451
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8530 - loss: 0.3586 - val_accuracy: 0.8687 - val_loss: 0.3405
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8561 - loss: 0.3535 - val_accuracy: 0.8700 - val_loss: 0.3368
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8495 - loss: 0.3499 - val_accuracy: 0.8700 - val_loss: 0.3348
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8592 - loss: 0.3357 - val_accuracy: 0.

In [13]:
# Step 10: Predict on test data
y_pred_prob = model.predict(X_test)

# Convert probabilities into 0 or 1
y_pred = (y_pred_prob > 0.5).astype(int).ravel()

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [14]:
# Step 11: Evaluate the model

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy Score:", accuracy * 100, "%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Accuracy Score: 86.35000000000001 %

Confusion Matrix:
[[1539   54]
 [ 219  188]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1593
           1       0.78      0.46      0.58       407

    accuracy                           0.86      2000
   macro avg       0.83      0.71      0.75      2000
weighted avg       0.86      0.86      0.85      2000



In [15]:
# Step 12: Identify improvements

print("\nModel Improvements:")
print("1. Added multiple hidden layers.")
print("2. Used Dropout to reduce overfitting.")
print("3. Used Adam optimizer for faster learning.")
print("4. Used Early Stopping to prevent unnecessary training.")
print("5. Used feature scaling and categorical encoding.")



Model Improvements:
1. Added multiple hidden layers.
2. Used Dropout to reduce overfitting.
3. Used Adam optimizer for faster learning.
4. Used Early Stopping to prevent unnecessary training.
5. Used feature scaling and categorical encoding.
